In [ ]:
# Step 4: EVI plausibility assessment

In [ ]:
# ===============================================================
# Step-4 — EVI plausibility (batch, no rasters, Landsat-only)
# Windows: fixed cadence (e.g., 5-yr) + custom hydrological epochs
# Natural-only validity (cultivated/urban/NoData excluded; ambiguous NAT kept)
# Multi-sensor fallback for EVI (L5/L7/L8) to avoid year gaps (e.g., 2000/2001)
# ===============================================================

import os
import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr
import rioxarray
import datacube
from datacube.utils.geometry import Geometry, CRS
from datacube.utils import masking
from shapely.geometry import mapping
from datetime import datetime
from rasterio.enums import Resampling
from rasterio import features

# -------------------
# CONFIG
# -------------------
START_YEAR   = 1988
END_YEAR     = 2023
INTERVAL_YRS = 5  # main cadence (re-run with 10 if desired)

# Hydrological epochs (edit as needed)
CUSTOM_EPOCHS = [
    {"label": "1988-1993_WetBaseline",        "start": 1988, "end": 1993},
    {"label": "1994-2000_EarlyDry",           "start": 1994, "end": 2000},
    {"label": "2001-2009_MillenniumDrought",  "start": 2001, "end": 2009},
    {"label": "2010-2012_PostDroughtFloods",  "start": 2010, "end": 2012},
    {"label": "2013-2017_ModerateDry",        "start": 2013, "end": 2017},
    {"label": "2018-2019_SevereDrought",      "start": 2018, "end": 2019},
    {"label": "2020-2022_LaNinaFloods",       "start": 2020, "end": 2022},
]

PIXEL_RES      = 30
PX_TO_HA       = (PIXEL_RES**2) / 10000.0
TARGET_CRS     = "EPSG:3577"
BUFFER_M       = 500

# WOfS persistent-water thresholds
WET_FREQ_THRESH = 80.0  # % wet of clear
MIN_CLEAR_OBS   = 20    # min clear obs to trust wetness frequency

# EVI plausibility thresholds (reporting only)
THRESH_EVI = {
    "relaxed": {"enc_devi_min": +0.02, "ret_devi_max": -0.02},
    "strict":  {"enc_devi_min": +0.05, "ret_devi_max": -0.05},
}
RUN_MODES = ["relaxed", "strict"]

# Seasonal window for EVI (Jul–Oct)
EVI_SEASON = ("07-01", "10-31")

BASE_DIR     = "/home/jovyan/DEA products paper/Data"
OUT_TABLES   = "/home/jovyan/DEA products paper/Results/step 5_evi_v2/tables_v3"
os.makedirs(OUT_TABLES, exist_ok=True)

# Subregion shapefiles (ANAE P1–P5)
WETLAND_SHP_PATHS = [
    os.path.join(BASE_DIR, "wetland boundaries/P1ANAEWetlands.shp"),
    os.path.join(BASE_DIR, "wetland boundaries/P2ANAEWetlands.shp"),
    os.path.join(BASE_DIR, "wetland boundaries/P3ANAEWetlands.shp"),
    os.path.join(BASE_DIR, "wetland boundaries/P4ANAEWetlands.shp"),
    os.path.join(BASE_DIR, "wetland boundaries/P5ANAEWetlands.shp"),
]

LC_PRODUCT = "ga_ls_landcover_class_cyear_3"  # backbone LC

# -------------------
# CLASS GROUPS (natural-only + ambiguous NAT kept)
# -------------------
WOODY_CODES = {20,27,28,29,30,31, 56,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77}
HERB_CODES  = {21,32,33,34,35,36, 57,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92}
BARE_CODES  = {94,95,96,97}
WATER_CODES = {98,99,100,101,102,103,104}
AMBIG_NAT   = {19,22,23,24,25,26, 55,58,59,60,61,62}

# Exclusions from validity
ARTIFICIAL_CODES = {93}
NODATA_CODES     = {255}
CULTIVATED       = {1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18}
MASK_OUT_CODES   = ARTIFICIAL_CODES | NODATA_CODES | CULTIVATED

# -------------------
# HELPERS
# -------------------
def make_periods(start, end, interval_years=None, custom_epochs=None):
    periods = []
    if interval_years:
        years = list(range(start, end, interval_years))
        for y0 in years:
            y1 = y0 + interval_years
            if y1 <= end:
                periods.append((f"{y0}-{y1}", y0, y1))
    if custom_epochs:
        for ep in custom_epochs:
            label = ep["label"]; y0, y1 = int(ep["start"]), int(ep["end"])
            if y0 < y1 and (start <= y0 < y1 <= end):
                periods.append((label, y0, y1))
    return periods

def dc_query_from_geom(geom, crs_str=TARGET_CRS, res=PIXEL_RES, buffer_m=BUFFER_M):
    geom_buf = geom.buffer(buffer_m)
    return {"geopolygon": Geometry(mapping(geom_buf), CRS(crs_str)),
            "output_crs": crs_str, "resolution": (-res, res)}

def mask_codes(da: xr.DataArray, keep: set) -> xr.DataArray:
    codes = np.array(list(keep), dtype="int32")
    return xr.apply_ufunc(np.isin, da.astype("int32"), codes,
                          dask="parallelized", output_dtypes=[bool])

def count_px(mask: xr.DataArray) -> int:
    return int(mask.sum().values)

def ensure_crs(da: xr.DataArray, crs=TARGET_CRS):
    if da.rio.crs is None:
        da = da.rio.write_crs(crs)
    return da

def reproj_to_match(da: xr.DataArray, match: xr.DataArray, resampling=Resampling.bilinear):
    return ensure_crs(da).rio.reproject(
        match.rio.crs,
        transform=match.rio.transform(recalc=True),
        shape=match.rio.shape,
        resampling=resampling
    )

def _autoscale_reflectance(da: xr.DataArray) -> xr.DataArray:
    v = da.astype("float32")
    if v.size > 0:
        s = v.isel(y=slice(0, min(50, v.sizes.get('y', 0))),
                   x=slice(0, min(50, v.sizes.get('x', 0))))
        if s.size > 0 and np.nanmedian(s.values) > 2.0:
            v = v / 10000.0
    return v.clip(0.0, 1.0)

def _season_bounds(year):
    return (f"{year}-{EVI_SEASON[0]}", f"{year}-{EVI_SEASON[1]}")

def evi_from_rgb(nir: xr.DataArray, red: xr.DataArray, blue: xr.DataArray) -> xr.DataArray:
    nir = _autoscale_reflectance(nir); red = _autoscale_reflectance(red); blue = _autoscale_reflectance(blue)
    denom = (nir + 6.0*red - 7.5*blue + 1.0)
    with np.errstate(divide="ignore", invalid="ignore"):
        evi = 2.5 * (nir - red) / denom
    return evi.clip(min=-1.0, max=1.0).astype("float32")

# ---- NEW: multi-sensor candidates & loader with fallback ----
def landsat_candidates_for_year(year):
    """
    Priority-ordered list of (product, resolution) to try for a given year.
    """
    if year >= 2015:
        return [("ga_ls8c_ard_3", (-30, 30)), ("ga_ls7e_ard_3", (-30, 30))]
    elif 2011 <= year <= 2014:
        return [("ga_ls7e_ard_3", (-30, 30)), ("ga_ls8c_ard_3", (-30, 30))]
    elif 1999 <= year <= 2010:
        # Try L5 first, then L7 (covers 2000/2001 gaps if L5 is empty)
        return [("ga_ls5t_ard_3", (-30, 30)), ("ga_ls7e_ard_3", (-30, 30))]
    elif 1988 <= year <= 1998:
        return [("ga_ls5t_ard_3", (-30, 30))]
    else:
        return []

def load_evi_year(dc, q, year, season=("07-01","10-31")):
    """
    Try multiple Landsat sensors for the given year; return seasonal median EVI.
    Returns (evi_da, product_used) or (None, None) if all candidates empty.
    """
    for prod, res in landsat_candidates_for_year(year):
        qp = dict(q); qp["resolution"] = res
        ds = dc.load(product=prod, time=(f"{year}-{season[0]}", f"{year}-{season[1]}"),
                     group_by="solar_day",
                     measurements=["nbart_blue","nbart_red","nbart_nir"], **qp)
        if ds.sizes.get("time", 0) > 0:
            comp = ds.median("time", skipna=True)
            evi = evi_from_rgb(comp["nbart_nir"], comp["nbart_red"], comp["nbart_blue"])
            return evi, prod
    return None, None

def auc_from_scores(scores, labels):
    s = np.asarray(scores).ravel(); y = np.asarray(labels).ravel()
    m = np.isfinite(s) & np.isfinite(y)
    s, y = s[m], y[m].astype(int)
    if y.sum()==0 or (1-y).sum()==0:
        return np.nan
    ranks = s.argsort().argsort().astype(float) + 1
    sum_ranks_pos = ranks[y==1].sum()
    n1 = (y==1).sum(); n0 = (y==0).sum()
    U = sum_ranks_pos - n1*(n1+1)/2
    return U / (n1*n0)

def summarize_evi_plausibility(trans_code, d_evi, mode):
    """
    trans_code: 0=masked, 1=NW->NW, 2=NW->W(enc), 3=W->NW(ret), 4=W->W
    Returns dict of support counts/percentages, medians, and AUCs.
    """
    T = THRESH_EVI[mode]
    enc = (trans_code == 2)
    ret = (trans_code == 3)
    stable = (trans_code == 1) | (trans_code == 4)

    enc_keep = (d_evi >= T["enc_devi_min"])
    ret_keep = (d_evi <= T["ret_devi_max"])

    def n(mask): return int(np.nansum(mask.values))
    def med(arr, mmask):
        vals = arr.where(mmask).values
        return float(np.nanmedian(vals)) if np.isfinite(vals).any() and n(mmask) > 0 else np.nan

    enc_n = n(enc); ret_n = n(ret)
    enc_ok = n(enc & enc_keep); ret_ok = n(ret & ret_keep)

    devi_enc_med = med(d_evi, enc)
    devi_ret_med = med(d_evi, ret)
    devi_st_med  = med(d_evi, stable)

    # AUC using ΔEVI directly (enc/ret vs stable)
    s_enc = d_evi.where(enc | stable).values
    y_enc = enc.where(enc | stable).fillna(False).values.astype(int)
    s_ret = d_evi.where(ret | stable).values
    y_ret = ret.where(ret | stable).fillna(False).values.astype(int)

    enc_auc = auc_from_scores(s_enc, y_enc)
    ret_auc = auc_from_scores(s_ret, y_ret)

    return {
        "Enc_px": enc_n, "Enc_supported_px": enc_ok,
        "Enc_support_%": (enc_ok/enc_n*100.0) if enc_n>0 else np.nan,
        "Ret_px": ret_n, "Ret_supported_px": ret_ok,
        "Ret_support_%": (ret_ok/ret_n*100.0) if ret_n>0 else np.nan,
        "ΔEVI_med_enc": devi_enc_med, "ΔEVI_med_ret": devi_ret_med, "ΔEVI_med_stable": devi_st_med,
        "AUC_enc_vs_stable": enc_auc, "AUC_ret_vs_stable": ret_auc,
    }

# -------------------
# MAIN (batch per subregion, cached EVI loads)
# -------------------
periods = make_periods(START_YEAR, END_YEAR, interval_years=INTERVAL_YRS, custom_epochs=CUSTOM_EPOCHS)
print("Total periods (fixed + epochs):", len(periods))

dc = datacube.Datacube(app="WPE_Step5_EVI_Batch_LandsatOnly")

# Load wetlands (merge subregions with IDs)
wetland_list = []
for sub_idx, path in enumerate(WETLAND_SHP_PATHS, start=1):
    gdf = gpd.read_file(path).to_crs(TARGET_CRS)
    if "OBJECTID" not in gdf.columns:
        raise ValueError(f"{path} has no OBJECTID field!")
    gdf["WetlandID"] = gdf["OBJECTID"].astype(int)
    gdf["SubRegion"] = f"SR{sub_idx}"
    wetland_list.append(gdf)
wetlands = gpd.GeoDataFrame(pd.concat(wetland_list, ignore_index=True), crs=TARGET_CRS)

rows = []

for subr in wetlands["SubRegion"].unique():
    sub_wetlands = wetlands[wetlands["SubRegion"] == subr].reset_index(drop=True)
    print(f"▶ Processing {subr} ... ({len(sub_wetlands)} wetlands)")

    # Build subregion query once
    geom_union = sub_wetlands.unary_union
    q = dc_query_from_geom(geom_union)

    # Land cover (backbone grid)
    ds_lc = dc.load(product=LC_PRODUCT,
                    measurements=["full_classification"],
                    time=(f"{START_YEAR}-01-01", f"{END_YEAR}-12-31"),
                    **q)
    if ds_lc.sizes.get("time", 0) == 0:
        print(f"  ⚠️ No LC data for {subr}; skip")
        continue
    lc = ds_lc["full_classification"]

    # WOfS for persistent-water mask
    wo = dc.load(product="ga_ls_wo_3",
                 measurements=["water"],
                 time=(f"{START_YEAR}-01-01", f"{END_YEAR}-12-31"),
                 group_by="solar_day",
                 **q)
    if wo.sizes.get("time", 0) > 0:
        water_da = wo["water"]
        wet   = masking.make_mask(water_da, wet=True)
        dry   = masking.make_mask(water_da, dry=True)
        clear = wet | dry
        wet_count   = wet.astype("int16").sum("time")
        clear_count = clear.astype("int16").sum("time")
        with np.errstate(divide="ignore", invalid="ignore"):
            wet_freq_pct = (wet_count.astype("float32") /
                            xr.where(clear_count > 0, clear_count, np.nan)) * 100.0
    else:
        wet_freq_pct, clear_count = None, None

    # Rasterized WetlandID mask on LC grid
    transform = lc.rio.transform()
    shape = lc.rio.shape
    wetid_mask = features.rasterize(
        [(geom, wid) for wid, geom in zip(sub_wetlands["WetlandID"], sub_wetlands.geometry)],
        out_shape=shape, transform=transform, fill=0, dtype="int32"
    )
    wetid_mask = xr.DataArray(wetid_mask, dims=("y","x"), coords={"y": lc.y, "x": lc.x})

    # Cache EVI by year for this subregion to avoid re-loading
    evi_cache = {}
    evi_prod_used = {}  # optional: track which product was used

    for label, y0, y1 in periods:
        # LC yearly slices
        base = lc.sel(time=lc.time.dt.year==y0).squeeze(drop=True)
        tgt  = lc.sel(time=lc.time.dt.year==y1).squeeze(drop=True)
        if base.size == 0 or tgt.size == 0:
            print(f"  ⏭️ {subr} {label}: LC year missing")
            continue

        # Valid LC codes: natural-only; exclude cultivated/urban/NoData
        valid_base = ~mask_codes(base, MASK_OUT_CODES)
        valid_tgt  = ~mask_codes(tgt,  MASK_OUT_CODES)
        valid = valid_base & valid_tgt

        # Woody vs non-woody
        base_w  = mask_codes(base, WOODY_CODES) & valid
        tgt_w   = mask_codes(tgt,  WOODY_CODES) & valid
        base_nw = (~base_w) & valid
        tgt_nw  = (~tgt_w)  & valid

        # Persistent-water mask on LC grid
        if wet_freq_pct is not None and clear_count is not None:
            wf = wet_freq_pct.rio.reproject_match(base, resampling=Resampling.nearest)
            cc = clear_count.rio.reproject_match(base, resampling=Resampling.nearest)
            water_mask = (wf > WET_FREQ_THRESH) & (cc >= MIN_CLEAR_OBS)
        else:
            water_mask = xr.zeros_like(base, dtype=bool)

        keep = (~water_mask) & (wetid_mask > 0)

        # Transition codes AFTER water mask:
        # 0=masked, 1=NW->NW, 2=NW->W(enc), 3=W->NW(ret), 4=W->W
        trans_code = xr.zeros_like(base, dtype="int8")
        trans_code = xr.where(base_nw & tgt_nw   & keep, 1, trans_code)
        trans_code = xr.where(base_nw & tgt_w    & keep, 2, trans_code)
        trans_code = xr.where(base_w  & tgt_nw   & keep, 3, trans_code)
        trans_code = xr.where(base_w  & tgt_w    & keep, 4, trans_code)

        # EVI seasonal medians for baseline & target with multi-sensor fallback
        if y0 not in evi_cache:
            evi_b, prod_b = load_evi_year(dc, q, y0, season=EVI_SEASON)
            if evi_b is None:
                print(f"  ⚠️ {subr} [{label}] missing EVI {y0} across candidates")
                continue
            evi_cache[y0] = evi_b; evi_prod_used[y0] = prod_b
        if y1 not in evi_cache:
            evi_t, prod_t = load_evi_year(dc, q, y1, season=EVI_SEASON)
            if evi_t is None:
                print(f"  ⚠️ {subr} [{label}] missing EVI {y1} across candidates")
                continue
            evi_cache[y1] = evi_t; evi_prod_used[y1] = prod_t

        evi_b = reproj_to_match(evi_cache[y0], base, resampling=Resampling.bilinear)
        evi_t = reproj_to_match(evi_cache[y1], base, resampling=Resampling.bilinear)
        d_evi = (evi_t - evi_b).astype("float32")

        # Per-wetland summaries
        for wid in sub_wetlands["WetlandID"]:
            mask_w = (wetid_mask == wid)

            enc_px = count_px((trans_code == 2).where(mask_w))
            ret_px = count_px((trans_code == 3).where(mask_w))
            st_px  = count_px(((trans_code == 1) | (trans_code == 4)).where(mask_w))
            valid_after = enc_px + ret_px + st_px

            t_w    = trans_code.where(mask_w)
            devi_w = d_evi.where(mask_w)

            for mode in RUN_MODES:
                M = summarize_evi_plausibility(t_w, devi_w, mode)
                rows.append({
                    "SubRegion": subr,
                    "WetlandID": int(wid),
                    "Period": label,
                    "BaselineYear": y0,
                    "TargetYear": y1,
                    "Mode": f"evi_{mode}",
                    # Context counts after mask
                    "Valid_px_after": valid_after,
                    "NW->W_px_after": enc_px,
                    "W->NW_px_after": ret_px,
                    "Stable_px_after": st_px,
                    # (optional) products actually used for EVI at endpoints
                    "EVI_product_base": evi_prod_used[y0],
                    "EVI_product_target": evi_prod_used[y1],
                    # EVI plausibility metrics
                    **M
                })

        print(f"  ✅ {subr} [{label}] EVI plausibility tallied.")

# -------------------
# EXPORT
# -------------------
summary = pd.DataFrame(rows).sort_values(
    ["SubRegion","WetlandID","BaselineYear","Mode"]
).reset_index(drop=True)

ts = datetime.now().strftime("%Y%m%d_%H%M%S")
csv_path = os.path.join(OUT_TABLES,
    f"evi_plausibility_batch_{START_YEAR}_{END_YEAR}_{INTERVAL_YRS}yr_plusEpochs_{ts}.csv")
summary.to_csv(csv_path, index=False)

print("✅ Exported EVI plausibility batch table to:", csv_path)
summary
